
# Create a ProtocolQC template from an XNAT reference session

This notebook converts the original Flywheel template-generation workflow to XNAT.

It performs the following steps:

1. Connects to XNAT using environment variables or securely prompted credentials.
2. Lists the scans in a selected reference session.
3. Downloads each scan's `DICOM` resource as a ZIP archive.
4. Extracts acquisition parameters using the ProtocolQC implementation in the `pipelines` repository.
5. Builds the approved-protocol JSON structure used by ProtocolQC.
6. Saves the JSON locally for review.
7. Optionally uploads it to a project-level XNAT resource, by default:
   - Resource label: `ProtocolQC`
   - Filename: `protocol-template.json`

## XNAT API paths used

- List session scans:
  `/data/projects/{project}/subjects/{subject}/experiments/{session}/scans`
- Download one scan's DICOM resource:
  `/data/projects/{project}/subjects/{subject}/experiments/{session}/scans/{scan}/resources/DICOM/files?format=zip`
- Upload the resulting project resource file:
  `/data/projects/{project}/resources/{resource}/files/{filename}`

The notebook does not store credentials in the output JSON.

> **Current limitation:** the ProtocolQC DICOM extractor is still a scaffold. It currently handles stable public DICOM attributes. Enhanced MR functional groups, Siemens private tags, Phoenix parsing, spectroscopy, phase-encoding polarity, and full Flywheel compatibility remain implementation TODOs.


## Detailed compatibility limitations

This notebook/workbook is currently an **XNAT access and template-assembly prototype**. It should not yet be treated as production-equivalent to the Flywheel ProtocolQC template generator.

The supplied Flywheel template generator reads DICOM through `run.return_dicom_dict(...)` and `run.combined_dicom_dicts(...)`. This notebook currently imports the simplified scaffold functions `read_dicom_headers(...)` and `combine_dicom_parameters(...)`, so it does **not** yet use the full extraction logic.

Important limitations to review before approving any generated template:

1. **Enhanced MR functional groups are not fully parsed.** Values stored in `(5200,9229)` and `(5200,9230)` may be missed.
2. **SOP Class handling is not equivalent.** Enhanced MR, MR spectroscopy, regular MR Image Storage, Phoenix report-like objects, and unsupported SOP classes are not handled exactly as in the Flywheel implementation.
3. **Derived-image filtering is not equivalent.** The Flywheel extractor skips `ImageType` values containing `DERIVED`; this workbook does not yet guarantee identical filtering.
4. **Siemens private tags are not fully parsed.** Fields such as `Sequence`, `Sequence_options`, and Phoenix-derived values may not match Flywheel output.
5. **Phase-encoding polarity is not implemented.** The Flywheel code derives `PE_polarity` from Phoenix protocol text; this workbook may leave it as `NA` or omit it.
6. **Spectroscopy fallbacks are incomplete.** The Flywheel code contains specific fallbacks for missing image-oriented enhanced MR structures.
7. **Calculated geometry fields may be missing or simplified.** Review `freq_fov`, `phase_fov`, `Freq_row`, `Freq_col`, `Phase_row`, `Phase_col`, `Slice_Gap`, `Freq_res`, and `Phase_res`.
8. **Multi-echo behaviour is only partial.** The Flywheel code turns differing values across files into lists and updates `Sequence_Attributes.Multi-Echo`.
9. **DICOM file-count multiplier semantics are not fully validated.** Manually review `Check_DICOM_File_Number`, `Temporal_positions_multiplier`, `Echo_lines_multiplier`, `Temporal_positions`, and `Multi-Echo`.
10. **Phoenix ZIP report handling is not implemented.** Template `SequenceList` completeness checks based on Phoenix ZIP reports are not yet reproduced.
11. **Flywheel-specific metadata and side effects are intentionally excluded.** Session tags, Twilio SMS, Flywheel project attachments, and `/flywheel/v0/output` paths should not be ported directly; XNAT-native replacements need separate design.
12. **XNAT resource assumptions must be validated.** Scan DICOM download, project-resource upload, labels, permissions, and resource naming must be tested in the local sandbox and target deployment.

**Do not approve a generated template until all `NA` values, missing fields, sequence alternatives, ordinal numbers, tolerances, and sequence attributes have been reviewed manually.**

The next implementation step is to port the Flywheel extraction logic into `protocol_qc/dicom.py` without Flywheel side effects, then make this notebook and the runtime ProtocolQC pipeline call that same shared extraction module.


In [ ]:

# Optional installation commands. Run only when the environment does not already
# contain these packages.
#
# %pip install requests pydicom
# %pip install -e "D:/repos/pipelines"


In [ ]:

from __future__ import annotations

import getpass
import json
import os
import shutil
import sys
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote

import requests

# Adjust this when running outside the editable ais-pipelines environment.
PIPELINES_REPO = Path(r"D:\repos\pipelines")
PIPELINES_SRC = PIPELINES_REPO / "src"

if PIPELINES_SRC.exists() and str(PIPELINES_SRC) not in sys.path:
    sys.path.insert(0, str(PIPELINES_SRC))

try:
    from australianimagingservice.quality_control.protocol_qc.dicom import (
        DICTIONARY_VERSION,
        combine_dicom_parameters,
        read_dicom_headers,
    )
except ImportError as error:
    raise ImportError(
        "Could not import ProtocolQC from the pipelines repository. "
        "Activate the ais-pipelines environment or update PIPELINES_REPO."
    ) from error

print(f"ProtocolQC dictionary version: {DICTIONARY_VERSION}")



## Configuration

Use the XNAT **project ID**, **subject label or ID**, and **session label or ID**.

Credentials are read from `XNAT_USER` and `XNAT_PASS` when available. Otherwise, the notebook prompts for them. The password prompt is hidden.


In [ ]:

# XNAT connection and reference-session settings
XNAT_URL = os.environ.get("XNAT_HOST", "http://localhost")
PROJECT_ID = "REPLACE_WITH_PROJECT_ID"
SUBJECT_ID = "REPLACE_WITH_SUBJECT_ID"
SESSION_ID = "REPLACE_WITH_SESSION_ID"

# Scan resource holding DICOM files.
DICOM_RESOURCE_LABEL = "DICOM"

# Destination project resource and filename.
OUTPUT_RESOURCE_LABEL = "ProtocolQC"
OUTPUT_FILENAME = "protocol-template.json"

# Local working/output location.
OUTPUT_DIRECTORY = Path.cwd() / "protocolqc-template-output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Upload is deliberately disabled until the generated file has been reviewed.
UPLOAD_TO_XNAT = False
OVERWRITE_EXISTING_TEMPLATE = False

# Default tolerances retained from the Flywheel template-generation script.
TOLERANCE_FIELDS: dict[str, float] = {
    "TR": 10,
    "TE": 1,
    "TI": 1,
    "Flip_angle": 0.01,
    "Bandwidth": 10,
    "Slice_thickness": 0.01,
    "Slice_spacing": 0.01,
}


In [ ]:

def require_setting(name: str, value: str) -> str:
    """Reject placeholder or empty configuration values."""

    if not value or value.startswith("REPLACE_WITH_"):
        raise ValueError(f"Set {name} before continuing")
    return value


require_setting("PROJECT_ID", PROJECT_ID)
require_setting("SUBJECT_ID", SUBJECT_ID)
require_setting("SESSION_ID", SESSION_ID)

username = os.environ.get("XNAT_USER") or input("XNAT username: ").strip()
password = os.environ.get("XNAT_PASS") or getpass.getpass("XNAT password: ")

if not username or not password:
    raise ValueError("XNAT username and password are required")

xnat = requests.Session()
xnat.auth = (username, password)
xnat.headers.update({"Accept": "application/json"})
XNAT_URL = XNAT_URL.rstrip("/")

# Test authentication and project visibility.
response = xnat.get(
    f"{XNAT_URL}/data/projects/{quote(PROJECT_ID, safe='')}",
    params={"format": "json"},
    timeout=60,
)
response.raise_for_status()
print(f"Connected to XNAT project: {PROJECT_ID}")


## XNAT helper functions

In [ ]:

def result_rows(response: requests.Response) -> list[dict[str, Any]]:
    """Return rows from a conventional XNAT ResultSet response."""

    payload = response.json()
    try:
        return list(payload["ResultSet"]["Result"])
    except (KeyError, TypeError) as error:
        raise RuntimeError(
            f"Unexpected XNAT response structure from {response.url}: {payload}"
        ) from error


def list_session_scans() -> list[dict[str, Any]]:
    """List scans for the configured XNAT session."""

    endpoint = (
        f"{XNAT_URL}/data/projects/{quote(PROJECT_ID, safe='')}"
        f"/subjects/{quote(SUBJECT_ID, safe='')}"
        f"/experiments/{quote(SESSION_ID, safe='')}/scans"
    )
    response = xnat.get(endpoint, params={"format": "json"}, timeout=60)
    response.raise_for_status()
    return result_rows(response)


def download_scan_dicom_zip(scan_id: str, destination: Path) -> Path:
    """Download one XNAT scan's DICOM resource as a ZIP archive."""

    endpoint = (
        f"{XNAT_URL}/data/projects/{quote(PROJECT_ID, safe='')}"
        f"/subjects/{quote(SUBJECT_ID, safe='')}"
        f"/experiments/{quote(SESSION_ID, safe='')}"
        f"/scans/{quote(str(scan_id), safe='')}"
        f"/resources/{quote(DICOM_RESOURCE_LABEL, safe='')}/files"
    )

    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".partial")

    try:
        with xnat.get(
            endpoint,
            params={"format": "zip"},
            stream=True,
            timeout=300,
            headers={"Accept": "application/zip"},
        ) as response:
            response.raise_for_status()
            with partial.open("wb") as output_file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output_file.write(chunk)
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise

    if not zipfile.is_zipfile(destination):
        raise RuntimeError(
            f"XNAT did not return a ZIP archive for scan {scan_id}: {destination}"
        )
    return destination


def upload_project_resource_file(
    local_path: Path,
    *,
    resource_label: str,
    filename: str,
    overwrite: bool = False,
) -> None:
    """Upload a file to an XNAT project-level resource."""

    endpoint = (
        f"{XNAT_URL}/data/projects/{quote(PROJECT_ID, safe='')}"
        f"/resources/{quote(resource_label, safe='')}"
        f"/files/{quote(filename, safe='')}"
    )
    params = {
        "inbody": "true",
        "overwrite": str(overwrite).lower(),
    }

    with local_path.open("rb") as input_file:
        response = xnat.put(
            endpoint,
            params=params,
            data=input_file,
            headers={"Content-Type": "application/json"},
            timeout=120,
        )
    response.raise_for_status()
    print(
        f"Uploaded {local_path.name} to project resource "
        f"{PROJECT_ID}/{resource_label}/{filename}"
    )


## Inspect the reference session

In [ ]:

scans = list_session_scans()
print(f"Found {len(scans)} scans in session {SESSION_ID}")

for scan in scans:
    print(
        f"scan={scan.get('ID')} | type={scan.get('type')} | "
        f"series_description={scan.get('series_description', '')} | "
        f"quality={scan.get('quality', '')}"
    )



## Build the protocol template

The generated structure preserves the Flywheel template format:

- `Metadata`
- one key per `series_description`
- one or more numbered approved protocol variants per sequence
- `Acquisition_Parameters` entries containing `value` and optional `tolerance`
- `Sequence_Attributes.Ordinal_Number`

Scans without readable DICOM data are recorded in `generation_warnings` and skipped.


In [ ]:

def values_equal(left: Any, right: Any) -> bool:
    """Compare generated protocol dictionaries deterministically."""

    return json.dumps(left, sort_keys=True, default=str) == json.dumps(
        right, sort_keys=True, default=str
    )


def add_tolerances(sequence_dictionary: dict[str, Any]) -> dict[str, Any]:
    """Convert acquired parameter values into approved-template entries."""

    acquisition_parameters = sequence_dictionary["Acquisition_Parameters"]
    for field, field_value in list(acquisition_parameters.items()):
        wrapped: dict[str, Any] = {"value": field_value}
        if field in TOLERANCE_FIELDS:
            wrapped["tolerance"] = TOLERANCE_FIELDS[field]
        acquisition_parameters[field] = wrapped
    return sequence_dictionary


def build_protocol_template(
    scan_rows: list[dict[str, Any]],
) -> tuple[dict[str, Any], list[str]]:
    """Generate a ProtocolQC template from all DICOM scans in the session."""

    sequence_list: list[str] = []
    warnings: list[str] = []
    template: dict[str, Any] = {}
    sequence_number = 0

    metadata = {
        "SequenceNumber": 0,
        "DictionaryVersion": DICTIONARY_VERSION,
        "Comments": (
            "Generated from XNAT reference session "
            f"{PROJECT_ID}/{SUBJECT_ID}/{SESSION_ID}. Review all values, "
            "tolerances, sequence attributes, sequence order, and duplicate "
            "series descriptions before approving this template."
        ),
        "SequenceList": sequence_list,
        "Source": {
            "System": "XNAT",
            "ProjectID": PROJECT_ID,
            "SubjectID": SUBJECT_ID,
            "SessionID": SESSION_ID,
            "GeneratedUTC": datetime.now(timezone.utc).isoformat(),
        },
    }
    template["Metadata"] = metadata

    with tempfile.TemporaryDirectory(prefix="protocolqc-template-") as temporary:
        temp_root = Path(temporary)

        for scan in scan_rows:
            scan_id = str(scan.get("ID", "")).strip()
            if not scan_id:
                warnings.append(f"Skipped scan without an ID: {scan}")
                continue

            scan_root = temp_root / f"scan-{scan_id}"
            archive = scan_root / "dicom.zip"
            extracted = scan_root / "dicom"

            print(f"Processing XNAT scan {scan_id}...")
            try:
                download_scan_dicom_zip(scan_id, archive)
                extracted.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(archive) as zip_file:
                    zip_file.extractall(extracted)

                candidate_paths = [
                    path for path in extracted.rglob("*") if path.is_file()
                ]
                if not candidate_paths:
                    raise RuntimeError("Downloaded DICOM archive was empty")

                # read_dicom_headers will reject unreadable files. Some XNAT ZIPs
                # may contain non-DICOM files, so pre-filter them individually.
                readable_headers = []
                for path in candidate_paths:
                    try:
                        readable_headers.extend(read_dicom_headers([path]))
                    except RuntimeError:
                        continue

                if not readable_headers:
                    raise RuntimeError("No readable DICOM headers were found")

                sequence_dictionary = combine_dicom_parameters(readable_headers)
                sequence_dictionary = add_tolerances(sequence_dictionary)
                series_description = sequence_dictionary["Acquisition_Parameters"][
                    "series_description"
                ]["value"]

                if not series_description or series_description == "NA":
                    series_description = (
                        scan.get("series_description")
                        or scan.get("type")
                        or f"SCAN_{scan_id}"
                    )
                    sequence_dictionary["Acquisition_Parameters"][
                        "series_description"
                    ]["value"] = series_description

                sequence_list.append(series_description)

                if series_description not in template:
                    protocol_number = 1
                else:
                    existing = template[series_description]
                    duplicate = any(
                        values_equal(sequence_dictionary, variant)
                        for variant in existing.values()
                    )
                    if duplicate:
                        print(
                            f"Skipping duplicate protocol variant for "
                            f"{series_description}"
                        )
                        continue
                    protocol_number = max(int(number) for number in existing) + 1

                sequence_number += 1
                sequence_dictionary["Sequence_Attributes"][
                    "Ordinal_Number"
                ] = sequence_number
                sequence_dictionary["Sequence_Attributes"][
                    "XNAT_Scan_ID"
                ] = scan_id

                template.setdefault(series_description, {})[
                    protocol_number
                ] = sequence_dictionary
                print(
                    f"Added {series_description}, protocol variant "
                    f"{protocol_number}"
                )

            except Exception as error:
                message = f"Scan {scan_id} was skipped: {error}"
                warnings.append(message)
                print(f"WARNING: {message}")

    metadata["SequenceNumber"] = sequence_number
    metadata["SequenceList"] = sequence_list
    if warnings:
        metadata["GenerationWarnings"] = warnings

    return template, warnings


protocol_template, generation_warnings = build_protocol_template(scans)
print(
    f"Generated {protocol_template['Metadata']['SequenceNumber']} "
    "protocol entries"
)


## Save and review the template

In [ ]:

local_output = OUTPUT_DIRECTORY / OUTPUT_FILENAME
local_output.write_text(
    json.dumps(protocol_template, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved template: {local_output}")
print(f"Warnings: {len(generation_warnings)}")

# Show a compact summary without printing the complete JSON.
for name, value in protocol_template.items():
    if name == "Metadata":
        continue
    print(f"{name}: {len(value)} protocol variant(s)")



### Mandatory manual review before upload

Review the generated JSON and confirm at least:

- `Metadata.DictionaryVersion`
- sequence count and `SequenceList`
- removal of scouts, localisers, derived images, and unwanted series
- duplicate sequence descriptions and their numbered protocol variants
- ordinal numbers
- tolerance values
- `Multi-Echo`
- `Echo_lines_multiplier`
- `Temporal_positions_multiplier`
- `Check_DICOM_File_Number`
- enhanced MR and spectroscopy extraction results
- Siemens private-tag values and phase-encoding polarity, once implemented

On Windows, open the output with:

```powershell
& 'C:\Program Files\Notepad++\notepad++.exe' '<path-to-protocol-template.json>'
```


## Upload the reviewed template to the XNAT project resource

In [ ]:

if UPLOAD_TO_XNAT:
    upload_project_resource_file(
        local_output,
        resource_label=OUTPUT_RESOURCE_LABEL,
        filename=OUTPUT_FILENAME,
        overwrite=OVERWRITE_EXISTING_TEMPLATE,
    )
else:
    print(
        "Upload disabled. Review the JSON, then set UPLOAD_TO_XNAT=True. "
        "Set OVERWRITE_EXISTING_TEMPLATE=True only when replacement is intended."
    )



## Verify the uploaded project resource

After upload, verify it through the XNAT UI or the following API path:

`/data/projects/{project-id}/resources/ProtocolQC/files/protocol-template.json`

The ProtocolQC pipeline should use the same resource label and filename as its launch parameters.


In [ ]:

def verify_uploaded_template() -> dict[str, Any]:
    endpoint = (
        f"{XNAT_URL}/data/projects/{quote(PROJECT_ID, safe='')}"
        f"/resources/{quote(OUTPUT_RESOURCE_LABEL, safe='')}"
        f"/files/{quote(OUTPUT_FILENAME, safe='')}"
    )
    response = xnat.get(endpoint, timeout=60)
    response.raise_for_status()
    uploaded = response.json()
    print(
        f"Verified project resource: "
        f"{OUTPUT_RESOURCE_LABEL}/{OUTPUT_FILENAME}"
    )
    return uploaded

# Run after upload:
# uploaded_template = verify_uploaded_template()


## Mandatory review checklist before upload

Before setting `UPLOAD_TO_XNAT = True`, check the generated JSON for:

- `Metadata.DictionaryVersion`
- `Metadata.SequenceNumber`
- `Metadata.SequenceList`
- duplicate sequence descriptions and numbered alternatives
- `Sequence_Attributes.Ordinal_Number`
- `Sequence_Attributes.Multi-Echo`
- `Sequence_Attributes.Check_DICOM_File_Number`
- `Sequence_Attributes.Temporal_positions_multiplier`
- `Sequence_Attributes.Echo_lines_multiplier`
- every tolerance value
- every `NA` value
- missing Siemens/Phoenix-derived fields
- missing calculated geometry fields
- unexpected derived images, scouts, reports, localisers, or optional sequences

The uploaded project-level template should be treated as approved only after this review is complete.
